# Zero-Label Defect Hunter – Development Notebook

This notebook trains an anomaly-detection model **using only good product images**.
No defective samples are used during training.

**Sections**
1. Imports
2. Dataset loading
3. Visualization
4. Preprocessing & feature extraction
5. Model training
6. Model saving
7. Testing
8. Anomaly score
9. Heatmap
10. Evaluation


## 1 — Imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

# Make project root importable
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from defect_hunter import DefectHunter, train_default_model

print("PyTorch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")


## 2 — Dataset loading (good images only)

In [ ]:
DATA = ROOT / "data"
TRAIN_GOOD = DATA / "train" / "good"
VAL_GOOD   = DATA / "validation" / "good"
TEST_GOOD  = DATA / "test" / "good"
TEST_DEF   = DATA / "test" / "defective"

train_files = sorted(TRAIN_GOOD.glob("*.jpg"))
val_files   = sorted(VAL_GOOD.glob("*.jpg"))
test_good   = sorted(TEST_GOOD.glob("*.jpg"))
test_def    = sorted(TEST_DEF.glob("*.jpg"))

print(f"Train (good): {len(train_files)}")
print(f"Val   (good): {len(val_files)}")
print(f"Test  (good): {len(test_good)}")
print(f"Test  (def):  {len(test_def)}")
assert len(train_files) > 0, "No training images found"


## 3 — Dataset visualization

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, p in zip(axes.flat, train_files[:10]):
    img = Image.open(p)
    ax.imshow(img)
    ax.set_title(p.name, fontsize=8)
    ax.axis("off")
plt.suptitle("Sample GOOD training images")
plt.tight_layout()
plt.show()


## 4 — Image preprocessing & feature extraction\nHandled inside `DefectHunter` (resize → ImageNet normalize → Wide-ResNet50 features).

In [ ]:
# Quick peek at a transformed tensor shape
hunter_tmp = DefectHunter(image_size=224)
img = Image.open(train_files[0]).convert("RGB")
t = hunter_tmp.transform(img)
print("Transformed tensor shape:", t.shape)
patches = hunter_tmp._extract_patches(t)
print("Patch feature matrix:", patches.shape)


## 5 — Model training (good images ONLY)

In [ ]:
hunter = DefectHunter(image_size=224, projection_dim=128)
hunter.fit(str(TRAIN_GOOD))


## 6 — Calibrate threshold & save model

In [ ]:
hunter.calibrate_threshold(str(VAL_GOOD), percentile=90.0)
MODEL_PATH = ROOT / "models" / "defect_hunter_model.pkl"
hunter.save(str(MODEL_PATH))
print("Stats:", hunter.stats)


## 7 — Testing on unseen images

In [ ]:
# One good and one defective example
good_path = str(test_good[0])
def_path  = str(test_def[0])

score_g, _ = hunter.predict(good_path, return_map=False)
score_d, _ = hunter.predict(def_path,  return_map=False)

print(f"Good image score : {score_g:.4f}  (threshold={hunter.threshold:.4f})")
print(f"Defective score  : {score_d:.4f}  (threshold={hunter.threshold:.4f})")


## 8 — Anomaly score summary

In [ ]:
print(f"Anomaly Score (defective): {score_d:.4f}")
print(f"Threshold:                 {hunter.threshold:.4f}")
print(f"Decision: {'SUSPICIOUS' if score_d >= hunter.threshold else 'NORMAL'}")


## 9 — Heatmap generation

In [ ]:
out_dir = ROOT / "results" / "predictions"
out_dir.mkdir(parents=True, exist_ok=True)

res = hunter.predict_and_visualize(def_path, str(out_dir), prefix="demo_defect")
print(res)

# Display
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, key, title in zip(axes, ["original", "heatmap", "overlay"],
                          ["Original", "Heatmap", "Overlay"]):
    ax.imshow(Image.open(res[key]))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 10 — Evaluation metrics

In [ ]:
y_true, y_score, y_pred = [], [], []

for p in test_good:
    s, _ = hunter.predict(str(p), return_map=False)
    y_true.append(0)
    y_score.append(s)
    y_pred.append(1 if s >= hunter.threshold else 0)

for p in test_def:
    s, _ = hunter.predict(str(p), return_map=False)
    y_true.append(1)
    y_score.append(s)
    y_pred.append(1 if s >= hunter.threshold else 0)

y_true = np.array(y_true)
y_score = np.array(y_score)
y_pred = np.array(y_pred)

print("Accuracy :", round(accuracy_score(y_true, y_pred), 3))
print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 3))
print("Recall   :", round(recall_score(y_true, y_pred, zero_division=0), 3))
print("F1       :", round(f1_score(y_true, y_pred, zero_division=0), 3))
try:
    print("ROC-AUC  :", round(roc_auc_score(y_true, y_score), 3))
except ValueError:
    print("ROC-AUC  : n/a (need both classes)")
